In [1]:
import pandas as pd

In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.metrics import root_mean_squared_log_error
from sklearn.model_selection import StratifiedKFold
from scipy.optimize import minimize

warnings.filterwarnings("ignore")
os.makedirs("models", exist_ok=True)

train = pd.read_csv("train.csv")
DROP = ["project_id", "hours_spent", "log_hours", "hours_bucket"]
features = [c for c in train.columns if c not in DROP]

y = np.log1p(train["hours_spent"].values)
h_true = train["hours_spent"].values
strata = train["hours_bucket"].fillna("unk").astype(str)

# 1. CatBoost feature matrix
X_cb = train[features].copy()
cat_cols = []
for c in features:
    if X_cb[c].dtype == bool or str(X_cb[c].dtype) == "boolean":
        X_cb[c] = X_cb[c].astype("float32")
    elif not pd.api.types.is_numeric_dtype(X_cb[c]):
        cat_cols.append(c)
        fill_v = "other" if c == "project_category" else ""
        X_cb[c] = X_cb[c].fillna(fill_v).astype(str)
    else:
        X_cb[c] = pd.to_numeric(X_cb[c], errors="coerce").astype("float32")

# 2. XGBoost feature matrix (pure string categories + float32 numerics/booleans)
X_xgb = train[features].copy()
bool_cols = [c for c in features if X_xgb[c].dtype == bool or str(X_xgb[c].dtype) == "boolean"]
for c in bool_cols:
    X_xgb[c] = pd.to_numeric(X_xgb[c].astype("float32"), errors="coerce")
xgb_cats = {}
for c in cat_cols:
    s = X_xgb[c].dropna().astype(str).reindex(X_xgb.index)
    X_xgb[c] = s.astype("category")
    xgb_cats[c] = list(X_xgb[c].cat.categories)
for c in features:
    if c not in bool_cols and c not in cat_cols:
        X_xgb[c] = pd.to_numeric(X_xgb[c], errors="coerce").astype("float32")

SEEDS = (42, 2026, 7)
oof_cb7 = np.zeros(len(train))
oof_cb8 = np.zeros(len(train))
oof_xgb = np.zeros(len(train))

idx = 0
for seed in SEEDS:
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    for fold, (tr_i, va_i) in enumerate(skf.split(X_cb, strata)):
        cb7 = CatBoostRegressor(
            iterations=2200, learning_rate=0.015, depth=7, l2_leaf_reg=4.0,
            subsample=0.8, bootstrap_type="Bernoulli", cat_features=cat_cols,
            random_seed=seed + fold, verbose=0, allow_writing_files=False
        ).fit(X_cb.iloc[tr_i], y[tr_i])
        cb7.save_model(f"models/cb7_{idx}.cbm")
        oof_cb7[va_i] += cb7.predict(X_cb.iloc[va_i]) / len(SEEDS)

        cb8 = CatBoostRegressor(
            iterations=2200, learning_rate=0.015, depth=8, l2_leaf_reg=6.0,
            subsample=0.8, bootstrap_type="Bernoulli", cat_features=cat_cols,
            random_seed=seed + fold + 100, verbose=0, allow_writing_files=False
        ).fit(X_cb.iloc[tr_i], y[tr_i])
        cb8.save_model(f"models/cb8_{idx}.cbm")
        oof_cb8[va_i] += cb8.predict(X_cb.iloc[va_i]) / len(SEEDS)

        xm = XGBRegressor(
            n_estimators=1600, learning_rate=0.012, max_depth=4, min_child_weight=30,
            subsample=0.8, colsample_bytree=0.5, reg_alpha=1.5, reg_lambda=8.0,
            enable_categorical=True, tree_method="hist",
            random_state=seed + fold, n_jobs=-1, verbosity=0
        ).fit(X_xgb.iloc[tr_i], y[tr_i])
        xm.save_model(f"models/xgb_{idx}.json")
        oof_xgb[va_i] += xm.predict(X_xgb.iloc[va_i]) / len(SEEDS)

        idx += 1
        print(f"Trained fold {idx}/15")

M = np.column_stack([oof_cb7, oof_cb8, oof_xgb])
res = minimize(
    lambda w: root_mean_squared_log_error(h_true, np.maximum(0.05, np.expm1(M @ (w / np.sum(w))))),
    [0.45, 0.45, 0.10], bounds=[(0, 1)] * 3
)
weights = (res.x / np.sum(res.x)).tolist()
print("Optimal weights [cb7, cb8, xgb]:", [round(w, 4) for w in weights])
print("3-Seed Out-of-Fold CV RMSLE:", round(res.fun, 6))

with open("models/meta.json", "w") as f:
    json.dump({
        "features": features,
        "cat_cols": cat_cols,
        "bool_cols": bool_cols,
        "xgb_cats": xgb_cats,
        "n_models": idx,
        "weights": weights,
    }, f)
print("Saved all 45 models + models/meta.json successfully!")
